# DEAP MFMC Scalability Experiment

This notebook implements the supplementary scalability experiment for MFMC on DEAP. The goal is to measure how the generalized leave-one-out MFMC objective behaves when moving from three modalities to four and five modalities, with compute and memory as the primary outcomes and best test accuracy as a compact secondary outcome.

The experiment reuses the existing DEAP subject-dependent MLP fusion baseline style: 5-fold stratified cross-validation, batch size 200, 20,001 total loop counter with 20,000 training iterations, Adam/AMSGrad, the same encoder and classifier modules, and the same downstream EEG classifier evaluation used in the original tri-modal DEAP notebook.


## 1. Experiment Goal

Settings implemented here:

1. 3-modality Full-sum MFMC: EEG + EOG + SKT
2. 4-modality Full-sum MFMC: EEG + EOG + SKT + GSR
3. 5-modality Full-sum MFMC: EEG + EOG + SKT + GSR + Respiration
4. 5-modality Sampled MFMC: EEG + EOG + SKT + GSR + Respiration with stochastic leave-one-out term sampling

For each setting, the notebook records mean best test accuracy across folds, training wall-clock time, time per iteration, peak GPU memory, model parameter count, and the number of MFMC terms evaluated per batch.


In [ ]:
# =============================================================================
# CONFIGURATION
# =============================================================================

import gc
import json
import math
import os
import pickle
import random
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from IPython.display import Markdown, display
from sklearn.model_selection import StratifiedKFold

PROJECT_ROOT = Path(os.environ.get('TAFFC_MFMC_ROOT', '/home/zhengdeyang/TAFFC_MFMC'))
MFMC_ROOT = PROJECT_ROOT / 'MFMC'
DATA_DIR = MFMC_ROOT / 'DEAP' / 'Data_processed'
OUT_DIR = Path(os.environ.get('MFMC_SCALABILITY_OUT_DIR', str(MFMC_ROOT / 'Supplement' / 'scalability_experiment')))
LOG_DIR = OUT_DIR / 'logs'
FIG_DIR = OUT_DIR / 'figures'

OUT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

PER_FOLD_RESULTS_CSV = OUT_DIR / 'deap_scalability_per_fold_results.csv'
AGGREGATE_RESULTS_CSV = OUT_DIR / 'deap_scalability_aggregate_results.csv'
COMPLEXITY_SUMMARY_CSV = OUT_DIR / 'deap_scalability_complexity_summary.csv'
MODEL_SIZE_SUMMARY_CSV = OUT_DIR / 'deap_scalability_model_size_summary.csv'

# Matches the existing DEAP MFMC MLP subject-dependent notebook by default.
BATCH_SIZE = int(os.environ.get('MFMC_SCALABILITY_BATCH_SIZE', '200'))
TOTAL_ITERATIONS = int(os.environ.get('MFMC_SCALABILITY_TOTAL_ITERATIONS', '20001'))
EVAL_INTERVAL = int(os.environ.get('MFMC_SCALABILITY_EVAL_INTERVAL', '500'))
N_FOLDS = int(os.environ.get('MFMC_SCALABILITY_N_FOLDS', '5'))
RANDOM_SEED = int(os.environ.get('MFMC_SCALABILITY_RANDOM_SEED', '42'))

LEARNING_RATE_ENCODER = float(os.environ.get('MFMC_SCALABILITY_ENCODER_LR', '0.0003'))
LEARNING_RATE_CLASSIFIER = float(os.environ.get('MFMC_SCALABILITY_CLASSIFIER_LR', '0.0003'))
BETA1 = 0.9
BETA2 = 0.999
COV_BETA = float(os.environ.get('MFMC_SCALABILITY_COV_BETA', '0.5'))
FEATURE_DIM = 128
PROJECTION_HIDDEN_DIM = 512
TEST_BATCH_SIZE = 100
USE_CLASS_BALANCING = False

# Sampled leave-one-out terms for the 5-modal sampled setting.
SAMPLED_TERMS_5MOD = int(os.environ.get('MFMC_SCALABILITY_SAMPLED_TERMS', '2'))

# Long training is deliberately guarded. Set this to True in the notebook, or run
# with MFMC_SCALABILITY_RUN=1, to launch the full experiment.
RUN_EXPERIMENTS = True
FORCE_RERUN = os.environ.get('MFMC_SCALABILITY_FORCE_RERUN', '0') == '1'
EXPERIMENT_FILTER = os.environ.get('MFMC_SCALABILITY_EXPERIMENT_FILTER', '').strip()
LOSS_LOG_INTERVAL = int(os.environ.get('MFMC_SCALABILITY_LOSS_LOG_INTERVAL', '100'))
PROGRESS_INTERVAL = int(os.environ.get('MFMC_SCALABILITY_PROGRESS_INTERVAL', '500'))

print('DEAP MFMC scalability configuration')
print('=' * 60)
print(f'Data directory: {DATA_DIR}')
print(f'Output directory: {OUT_DIR}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Total loop counter: {TOTAL_ITERATIONS} ({TOTAL_ITERATIONS - 1} training iterations)')
print(f'Evaluation interval: {EVAL_INTERVAL}')
print(f'5-modal sampled terms per batch: {SAMPLED_TERMS_5MOD}')
print(f'RUN_EXPERIMENTS: {RUN_EXPERIMENTS}')


## 2. Data Loading and Modality Mapping

The processed DEAP tensors are read from `MFMC/DEAP/Data_processed`. The preprocessing script documents the modality mapping as follows: EEG uses raw channels 0-31, EOG uses hEOG and vEOG, SKT is the skin temperature channel, GSR is raw channel 36, and respiration is raw channel 38. `skt_data.npy` and `temp_data.npy` are aliases for the same SKT/temperature signal; this notebook prefers `skt_data.npy` and falls back to `temp_data.npy` only if needed.


In [ ]:
# =============================================================================
# DATA LOADING AND MODALITY MAPPING
# =============================================================================

MODALITY_DEFINITIONS = {
    'eeg': {
        'label': 'EEG',
        'candidates': ['eeg_data.npy'],
        'note': 'DEAP raw channels 0-31',
    },
    'eog': {
        'label': 'EOG',
        'candidates': ['eog_data.npy'],
        'note': 'hEOG and vEOG, raw channels 32-33',
    },
    'skt': {
        'label': 'SKT',
        'candidates': ['skt_data.npy', 'temp_data.npy'],
        'note': 'skin temperature; skt_data.npy is an alias of temp_data.npy',
    },
    'gsr': {
        'label': 'GSR',
        'candidates': ['gsr_data.npy'],
        'note': 'raw channel 36, per-window z-score normalization',
    },
    'resp': {
        'label': 'Respiration',
        'candidates': ['resp_data.npy'],
        'note': 'raw channel 38, per-window z-score normalization',
    },
}

EXPERIMENTS = [
    {
        'setting_id': '3mod_full',
        'setting_label': '3-mod Full-sum',
        'modalities': ['eeg', 'eog', 'skt'],
        'mfmc_mode': 'full',
        'sampled_terms': None,
    },
    {
        'setting_id': '4mod_full',
        'setting_label': '4-mod Full-sum',
        'modalities': ['eeg', 'eog', 'skt', 'gsr'],
        'mfmc_mode': 'full',
        'sampled_terms': None,
    },
    {
        'setting_id': '5mod_full',
        'setting_label': '5-mod Full-sum',
        'modalities': ['eeg', 'eog', 'skt', 'gsr', 'resp'],
        'mfmc_mode': 'full',
        'sampled_terms': None,
    },
    {
        'setting_id': '5mod_sampled',
        'setting_label': '5-mod Sampled',
        'modalities': ['eeg', 'eog', 'skt', 'gsr', 'resp'],
        'mfmc_mode': 'sampled',
        'sampled_terms': SAMPLED_TERMS_5MOD,
    },
]


def load_numpy_tensor(candidates):
    for filename in candidates:
        path = DATA_DIR / filename
        if path.exists():
            return filename, torch.from_numpy(np.load(path)).float()
    raise FileNotFoundError(f'None of these files were found in {DATA_DIR}: {candidates}')


print('Loading processed DEAP tensors...')
subject = torch.from_numpy(np.load(DATA_DIR / 'subject.npy')).long()
emotion_labels = torch.from_numpy(np.load(DATA_DIR / 'emotion_labels.npy')).long()
labels_np = emotion_labels.numpy()

modality_tensors = {}
modality_rows = []
for modality_id, spec in MODALITY_DEFINITIONS.items():
    filename, tensor = load_numpy_tensor(spec['candidates'])
    modality_tensors[modality_id] = tensor
    modality_rows.append({
        'modality_id': modality_id,
        'modality': spec['label'],
        'file_used': filename,
        'shape': tuple(tensor.shape),
        'dtype': str(tensor.dtype),
        'note': spec['note'],
    })

mapping_df = pd.DataFrame(modality_rows)
display(mapping_df)

sample_count = len(emotion_labels)
for modality_id, tensor in modality_tensors.items():
    assert tensor.shape[0] == sample_count, f'{modality_id} sample count does not match labels'

class_counts = torch.bincount(emotion_labels)
print(f'Total samples: {sample_count}')
print(f'Unique subjects: {len(torch.unique(subject))}')
print(f'Class counts: {class_counts.tolist()}')

indices = np.arange(sample_count)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)
fold_splits = []
for fold_idx, (train_indices, test_indices) in enumerate(skf.split(indices, labels_np), start=1):
    fold_splits.append({
        'fold': fold_idx,
        'train_indices': train_indices,
        'test_indices': test_indices,
        'train_size': len(train_indices),
        'test_size': len(test_indices),
    })
    print(f'Fold {fold_idx}: train={len(train_indices)}, test={len(test_indices)}')


## 3. Reused Baseline Code and Implementation Notes

The model definitions below are copied in spirit from `MFMC/DEAP/MFMC_Fusion_MLP/DEAP_MFMC_Fusion_MLP_subject_dep.ipynb`:

- the per-modality encoder is the same `Advanced1DCNN_channel` pipeline,
- the projection head keeps the same two-layer MLP pattern with BatchNorm,
- the classifier is the same four-layer MLP classifier,
- the optimizer family and learning rates match the existing DEAP MLP notebook,
- the classifier is trained and evaluated on EEG features, matching the original DEAP subject-dependent MFMC setup.

The only structural generalization is that the projection head input grows from `(M - 1) * 128` instead of the original fixed `2 * 128` tri-modal input.


In [ ]:
# =============================================================================
# DEVICE SELECTION AND REUSED MODEL COMPONENTS
# =============================================================================


def select_device():
    """Select CUDA when available; otherwise fall back to CPU."""
    if not torch.cuda.is_available():
        print('CUDA is not available. Running on CPU; GPU memory fields will be NaN.')
        return torch.device('cpu')

    try:
        query = [
            'nvidia-smi',
            '--query-gpu=index,memory.free,memory.total,memory.used',
            '--format=csv,noheader,nounits',
        ]
        output = subprocess.check_output(query, encoding='utf-8')
        gpu_stats = []
        for line in output.strip().splitlines():
            index, free_mem, total_mem, used_mem = [part.strip() for part in line.split(',')]
            gpu_stats.append({
                'index': int(index),
                'free': int(free_mem),
                'total': int(total_mem),
                'used': int(used_mem),
            })
        selected = max(gpu_stats, key=lambda gpu: gpu['free'])
        device = torch.device(f"cuda:{selected['index']}")
        print(
            f"Using device: {device} "
            f"({selected['free']} MB free / {selected['total']} MB total, {selected['used']} MB used)"
        )
        return device
    except Exception as exc:
        print(f'Could not query nvidia-smi ({exc}). Falling back to cuda:0.')
        return torch.device('cuda:0')


def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


class NETWORK_F_MLP(nn.Module):
    """Multi-layer perceptron for feature transformation."""
    def __init__(self, input_dim=784, hidden_dim=200, out_dim=200, num_layers=2):
        super(NETWORK_F_MLP, self).__init__()
        self.dim = out_dim
        self.num_layers = num_layers

        self.fc_list = []
        self.bn_list = []

        self.fc_list.append(nn.Linear(input_dim, hidden_dim, bias=True))
        self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        for _ in range(self.num_layers - 1):
            self.fc_list.append(nn.Linear(hidden_dim, hidden_dim, bias=True))
            self.bn_list.append(nn.BatchNorm1d(hidden_dim))

        self.fc_list = nn.ModuleList(self.fc_list)
        self.bn_list = nn.ModuleList(self.bn_list)
        self.fc_final = nn.Linear(hidden_dim, out_dim, bias=True)

    def forward(self, x):
        x = x.reshape(x.shape[0], -1)
        for i in range(self.num_layers):
            x = self.fc_list[i](x)
            x = torch.relu(x)
            x = self.bn_list[i](x)
        x = self.fc_final(x)
        x = torch.sigmoid(x)
        return x


class Advanced1DCNN_channel(nn.Module):
    """1D CNN encoder reused from the existing DEAP MFMC MLP notebook."""
    def __init__(self, input_channels=1, num_classes=128, input_size=1280):
        super(Advanced1DCNN_channel, self).__init__()

        self.conv1 = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=11, padding=5),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv2 = nn.Sequential(
            nn.Conv1d(32, 64, kernel_size=11, padding=5),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv3 = nn.Sequential(
            nn.Conv1d(64, 128, kernel_size=11, padding=5),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )
        self.conv4 = nn.Sequential(
            nn.Conv1d(128, 256, kernel_size=11, padding=5),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=4, stride=4),
        )

        feat_size = input_size // (4 * 4 * 4 * 4)
        self.fc1 = nn.Sequential(
            nn.Linear(256 * feat_size, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
        )
        self.fc2 = nn.Sequential(
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
        )
        self.fc3 = nn.Linear(512, num_classes)

        self.MLP = NETWORK_F_MLP(
            input_dim=128 * input_channels,
            hidden_dim=4000,
            out_dim=num_classes,
            num_layers=1,
        )

    def forward(self, x):
        batch_size, channels = x.shape[0], x.shape[1]
        x = x.unsqueeze(2)
        x = x.flatten(0, 1)

        out = self.conv1(x)
        out = self.conv2(out)
        out = self.conv3(out)
        out = self.conv4(out)
        out = out.view(out.size(0), -1)
        out = self.fc1(out)
        out = self.fc2(out)
        out = self.fc3(out)

        out = out.reshape(batch_size, channels, -1)
        out = out.flatten(-2, -1)
        out = self.MLP(out)
        return out


class ComplexClassifier(nn.Module):
    """Classifier reused from the existing DEAP MFMC MLP notebook."""
    def __init__(self, dim_features=128, num_classes=4):
        super(ComplexClassifier, self).__init__()
        self.fc1 = nn.Linear(dim_features, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2 = nn.Linear(256, 512)
        self.bn2 = nn.BatchNorm1d(512)
        self.fc3 = nn.Linear(512, 256)
        self.bn3 = nn.BatchNorm1d(256)
        self.fc4 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = torch.relu(self.bn1(self.fc1(x)))
        x = torch.relu(self.bn2(self.fc2(x)))
        x = torch.relu(self.bn3(self.fc3(x)))
        x = self.fc4(x)
        return x


class ProjectionHead(nn.Module):
    """
    Two-layer MLP projection head for MFMC.

    For three modalities, input_dim is 256 and this matches the existing code.
    For more modalities, input_dim becomes (M - 1) * 128.
    """
    def __init__(self, input_dim=256, hidden_dim=512, output_dim=128):
        super(ProjectionHead, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.bn1 = nn.BatchNorm1d(hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        self.bn2 = nn.BatchNorm1d(output_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn2(x)
        return x


def count_parameters(*modules):
    return int(sum(p.numel() for module in modules for p in module.parameters()))


## 4. Generalized MFMC Objective for M Modalities

For a batch with `M` modality features `f_1, ..., f_M`, the full-sum objective computes every leave-one-out term:

`L_full = sum_i MFMC(f_i, P_i(concat({f_j : j != i})))`.

This is exactly the original tri-modal pattern when `M = 3`: each modality is correlated against a projection of the other two modalities. The generalization below keeps one projection head per target modality and one covariance tracker per target modality.


In [ ]:
# =============================================================================
# GENERALIZED MFMC LOSS
# =============================================================================


def adaptive_estimation(v_t, beta, square_term, update_step):
    """Adaptive smoothing filter for covariance tracking."""
    v_t = beta * v_t + (1 - beta) * square_term.detach()
    correction = 1 - beta ** max(update_step, 1)
    return v_t, v_t / correction


def MFMC_t_trace(x, y, track_cov, cov_beta=0.95):
    """Compute the MFMC-T trace loss for one target/projection pair."""
    Rx = (x.T @ x) / x.shape[0]
    Ry = (y.T @ y) / y.shape[0]
    Pxy = (x.T @ y) / x.shape[0]

    eps = 1e-6
    eye = torch.eye(Rx.shape[0], device=Rx.device, dtype=Rx.dtype)
    Rx = Rx + eye * eps
    Ry = Ry + eye * eps

    update_step = int(track_cov.get('updates', 0)) + 1
    track_cov['updates'] = update_step
    track_cov['Rx'], Rx_est = adaptive_estimation(track_cov['Rx'], cov_beta, Rx, update_step)
    track_cov['Ry'], Ry_est = adaptive_estimation(track_cov['Ry'], cov_beta, Ry, update_step)
    track_cov['Pxy'], Pxy_est = adaptive_estimation(track_cov['Pxy'], cov_beta, Pxy, update_step)

    Rx_est_inv = torch.inverse(Rx_est)
    Ry_est_inv = torch.inverse(Ry_est)

    cost = -Rx_est_inv @ Rx @ Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy @ Ry_est_inv @ Pxy_est.T \
           - Rx_est_inv @ Pxy_est @ Ry_est_inv @ Ry @ Ry_est_inv @ Pxy_est.T \
           + Rx_est_inv @ Pxy_est @ Ry_est_inv @ Pxy.T

    loss = -torch.trace(cost)
    return track_cov, loss


def create_covariance_trackers(modality_ids, feature_dim, device):
    trackers = {}
    for modality_id in modality_ids:
        trackers[modality_id] = {
            'Rx': torch.zeros(feature_dim, feature_dim, device=device),
            'Ry': torch.zeros(feature_dim, feature_dim, device=device),
            'Pxy': torch.zeros(feature_dim, feature_dim, device=device),
            'updates': 0,
        }
    return trackers


def generalized_projection_loss(
    features,
    projection_heads,
    trackers,
    modality_ids,
    cov_beta=0.5,
    sampled_terms=None,
    term_rng=None,
):
    """
    Generalize the tri-modal leave-one-out MFMC loss to M modalities.

    Full-sum mode evaluates all M target terms. Sampled mode evaluates a
    stochastic subset and scales by M / subset_size to approximate the full sum.
    """
    num_modalities = len(modality_ids)
    all_indices = list(range(num_modalities))

    if sampled_terms is None or sampled_terms >= num_modalities:
        selected_indices = all_indices
        scale = 1.0
    else:
        if term_rng is None:
            term_rng = random
        subset_size = max(1, int(sampled_terms))
        selected_indices = sorted(term_rng.sample(all_indices, subset_size))
        scale = num_modalities / subset_size

    losses = []
    selected_modalities = []
    for target_idx in selected_indices:
        target_id = modality_ids[target_idx]
        complement_features = [
            features[modality_ids[j]]
            for j in all_indices
            if j != target_idx
        ]
        complement = torch.cat(complement_features, dim=1)
        projected = projection_heads[target_id](complement)
        trackers[target_id], term_loss = MFMC_t_trace(
            features[target_id], projected, trackers[target_id], cov_beta=cov_beta
        )
        losses.append(term_loss)
        selected_modalities.append(target_id)

    total_loss = torch.stack(losses).sum() * scale
    return trackers, total_loss, selected_modalities, scale


## 5. 5-Modality Sampled MFMC

For the sampled 5-modal setting, each batch samples `s` leave-one-out terms from the five available target terms. The sampled objective is:

`L_sampled = (M / s) * sum_{i in S} MFMC(f_i, P_i(concat({f_j : j != i})))`, where `S` is a uniformly sampled subset of size `s`.

This scaling makes the sampled objective an unbiased estimator of the full-sum objective under uniform term sampling. In this notebook the default is `s = 2`, so the 5-modal sampled run evaluates two MFMC terms per batch instead of five, while still computing all modality encoders so that the downstream and fusion setup remains comparable.


## 6. Training and Evaluation Protocol

The training loop keeps the two-phase pattern from the existing DEAP notebook:

1. train all modality encoders and projection heads using the MFMC objective,
2. train the EEG classifier using cross-entropy while the EEG encoder is frozen for that classifier step,
3. evaluate test accuracy every `EVAL_INTERVAL` iterations,
4. keep only the best test accuracy observed during training for the final accuracy report.

The fold results are saved after every completed fold, so interrupted runs can resume without discarding finished work.


In [ ]:
# =============================================================================
# TRAINING, EVALUATION, AND RESULT LOGGING
# =============================================================================


def setting_modalities_label(setting):
    return ' + '.join(MODALITY_DEFINITIONS[mid]['label'] for mid in setting['modalities'])


def mfmc_terms_per_batch(setting):
    num_modalities = len(setting['modalities'])
    if setting['mfmc_mode'] == 'sampled':
        return min(int(setting['sampled_terms']), num_modalities)
    return num_modalities


def classification_modality(setting):
    # The original DEAP MFMC notebook trains and evaluates the classifier on EEG.
    return 'eeg' if 'eeg' in setting['modalities'] else setting['modalities'][0]


def create_models_for_setting(setting, device):
    modality_ids = setting['modalities']
    encoders = nn.ModuleDict()
    for modality_id in modality_ids:
        tensor = modality_tensors[modality_id]
        encoders[modality_id] = Advanced1DCNN_channel(
            input_channels=tensor.shape[1],
            num_classes=FEATURE_DIM,
            input_size=tensor.shape[2],
        )
    encoders = encoders.to(device)

    projection_input_dim = (len(modality_ids) - 1) * FEATURE_DIM
    projection_heads = nn.ModuleDict({
        modality_id: ProjectionHead(
            input_dim=projection_input_dim,
            hidden_dim=PROJECTION_HIDDEN_DIM,
            output_dim=FEATURE_DIM,
        )
        for modality_id in modality_ids
    }).to(device)

    num_classes = len(torch.unique(emotion_labels))
    classifier = ComplexClassifier(dim_features=FEATURE_DIM, num_classes=num_classes).to(device)
    return encoders, projection_heads, classifier


def evaluate_accuracy(encoder, classifier, test_tensor, test_labels, device):
    encoder.eval()
    classifier.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for start_idx in range(0, len(test_tensor), TEST_BATCH_SIZE):
            end_idx = min(start_idx + TEST_BATCH_SIZE, len(test_tensor))
            inputs = test_tensor[start_idx:end_idx].to(device)
            labels = test_labels[start_idx:end_idx].to(device)
            outputs = classifier(encoder(inputs))
            predicted = torch.argmax(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    encoder.train()
    classifier.train()
    return correct / total if total else 0.0


def cuda_sync(device):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)


def reset_peak_memory(device):
    if device.type == 'cuda':
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats(device)


def get_peak_memory_mb(device):
    if device.type != 'cuda':
        return math.nan, math.nan
    allocated = torch.cuda.max_memory_allocated(device) / (1024 ** 2)
    reserved = torch.cuda.max_memory_reserved(device) / (1024 ** 2)
    return float(allocated), float(reserved)


def load_per_fold_results():
    if PER_FOLD_RESULTS_CSV.exists():
        return pd.read_csv(PER_FOLD_RESULTS_CSV)
    return pd.DataFrame()


def write_aggregate_results(per_fold_df):
    if per_fold_df.empty:
        return pd.DataFrame()

    group_cols = [
        'setting_id',
        'setting_label',
        'modalities',
        'num_modalities',
        'mfmc_mode',
        'sampled_terms_requested',
        'mfmc_terms_per_batch',
        'model_parameter_count',
    ]

    agg = per_fold_df.groupby(group_cols, dropna=False).agg(
        folds_completed=('fold', 'count'),
        mean_best_test_accuracy=('best_test_accuracy', 'mean'),
        std_best_test_accuracy=('best_test_accuracy', lambda x: float(np.std(x, ddof=0))),
        mean_training_time_sec=('training_time_sec', 'mean'),
        std_training_time_sec=('training_time_sec', lambda x: float(np.std(x, ddof=0))),
        mean_sec_per_iteration=('sec_per_iteration', 'mean'),
        std_sec_per_iteration=('sec_per_iteration', lambda x: float(np.std(x, ddof=0))),
        mean_peak_gpu_memory_allocated_mb=('peak_gpu_memory_allocated_mb', 'mean'),
        max_peak_gpu_memory_allocated_mb=('peak_gpu_memory_allocated_mb', 'max'),
        mean_peak_gpu_memory_reserved_mb=('peak_gpu_memory_reserved_mb', 'mean'),
        max_peak_gpu_memory_reserved_mb=('peak_gpu_memory_reserved_mb', 'max'),
    ).reset_index()

    agg['mean_training_time_min'] = agg['mean_training_time_sec'] / 60.0
    agg['std_training_time_min'] = agg['std_training_time_sec'] / 60.0
    agg['best_test_accuracy_mean_pm_std'] = agg.apply(
        lambda row: f"{row['mean_best_test_accuracy']:.4f} +/- {row['std_best_test_accuracy']:.4f}",
        axis=1,
    )
    agg.to_csv(AGGREGATE_RESULTS_CSV, index=False)
    return agg


def save_fold_log(setting, fold_num, payload):
    log_path = LOG_DIR / f"{setting['setting_id']}_fold_{fold_num}_log.json"
    with open(log_path, 'w') as f:
        json.dump(payload, f, indent=2)
    return log_path


def completed_fold_exists(existing_df, setting, fold_num):
    if FORCE_RERUN or existing_df.empty:
        return False

    mask = (
        (existing_df['setting_id'] == setting['setting_id'])
        & (existing_df['fold'] == fold_num)
        & (existing_df['total_iterations'] == TOTAL_ITERATIONS)
        & (existing_df['batch_size'] == BATCH_SIZE)
        & (existing_df['eval_interval'] == EVAL_INTERVAL)
    )
    return bool(mask.any())


def run_one_fold(setting, fold_info, device):
    fold_num = fold_info['fold']
    modality_ids = setting['modalities']
    train_indices = fold_info['train_indices']
    test_indices = fold_info['test_indices']
    fold_seed = RANDOM_SEED + 1000 * EXPERIMENTS.index(setting) + fold_num
    set_all_seeds(fold_seed)
    term_rng = random.Random(fold_seed + 17)

    train_tensors = {mid: modality_tensors[mid][train_indices] for mid in modality_ids}
    test_tensors = {mid: modality_tensors[mid][test_indices] for mid in modality_ids}
    train_labels = emotion_labels[train_indices]
    test_labels = emotion_labels[test_indices]

    train_class_counts = torch.bincount(train_labels)
    if USE_CLASS_BALANCING:
        class_weights = 1.0 / train_class_counts.clamp_min(1).float()
        class_weights = class_weights / class_weights.sum() * len(class_weights)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    else:
        criterion = nn.CrossEntropyLoss()

    encoders, projection_heads, classifier = create_models_for_setting(setting, device)
    model_parameter_count = count_parameters(encoders, projection_heads, classifier)

    optimizer_features = optim.Adam(
        list(encoders.parameters()) + list(projection_heads.parameters()),
        lr=LEARNING_RATE_ENCODER,
        betas=(BETA1, BETA2),
        amsgrad=True,
    )
    optimizer_classifier = optim.Adam(
        classifier.parameters(),
        lr=LEARNING_RATE_CLASSIFIER,
        betas=(BETA1, BETA2),
        amsgrad=True,
    )

    trackers = create_covariance_trackers(modality_ids, FEATURE_DIM, device)
    classifier_mid = classification_modality(setting)
    sampled_terms = setting['sampled_terms'] if setting['mfmc_mode'] == 'sampled' else None

    fold_costs = []
    fold_classifier_losses = []
    fold_test_accuracies = []
    best_accuracy = 0.0
    selected_term_counts = []

    print('=' * 70)
    print(f"{setting['setting_label']} | Fold {fold_num}/{N_FOLDS}")
    print(f"Modalities: {setting_modalities_label(setting)}")
    print(f"Train samples: {len(train_indices)}, Test samples: {len(test_indices)}")
    print(f"MFMC terms per batch: {mfmc_terms_per_batch(setting)}")
    print('=' * 70)

    reset_peak_memory(device)
    cuda_sync(device)
    fold_start_time = time.perf_counter()

    for iteration in range(1, TOTAL_ITERATIONS):
        # Phase 1: unsupervised encoder/projection training with generalized MFMC.
        optimizer_features.zero_grad()
        batch_indices = torch.randint(0, len(train_labels), (BATCH_SIZE,))
        inputs = {mid: train_tensors[mid][batch_indices].to(device) for mid in modality_ids}
        features = {mid: encoders[mid](inputs[mid]) for mid in modality_ids}

        trackers, mfmc_loss, selected_modalities, scale = generalized_projection_loss(
            features,
            projection_heads,
            trackers,
            modality_ids,
            cov_beta=COV_BETA,
            sampled_terms=sampled_terms,
            term_rng=term_rng,
        )
        mfmc_loss.backward()
        optimizer_features.step()
        selected_term_counts.append(len(selected_modalities))

        # Phase 2: supervised EEG classifier training, matching the existing code path.
        optimizer_classifier.zero_grad()
        batch_indices = torch.randint(0, len(train_labels), (BATCH_SIZE,))
        classifier_inputs = train_tensors[classifier_mid][batch_indices].to(device)
        labels_batch = train_labels[batch_indices].to(device)
        with torch.no_grad():
            classifier_features = encoders[classifier_mid](classifier_inputs)
        output_class = classifier(classifier_features.detach())
        classifier_loss = criterion(output_class, labels_batch)
        classifier_loss.backward()
        optimizer_classifier.step()

        if iteration == 1 or iteration % LOSS_LOG_INTERVAL == 0:
            fold_costs.append({'iteration': iteration, 'mfmc_loss': float(mfmc_loss.item())})
            fold_classifier_losses.append({'iteration': iteration, 'classifier_loss': float(classifier_loss.item())})

        if iteration % PROGRESS_INTERVAL == 0:
            print(
                f"{setting['setting_id']} fold {fold_num} iter {iteration:5d} | "
                f"MFMC: {mfmc_loss.item():.6f} | Classifier: {classifier_loss.item():.6f}"
            )

        if iteration % EVAL_INTERVAL == 0:
            accuracy = evaluate_accuracy(
                encoders[classifier_mid],
                classifier,
                test_tensors[classifier_mid],
                test_labels,
                device,
            )
            fold_test_accuracies.append({'iteration': iteration, 'test_accuracy': float(accuracy)})
            best_accuracy = max(best_accuracy, accuracy)
            print(f"{setting['setting_id']} fold {fold_num} eval iter {iteration:5d} | accuracy: {accuracy:.4f}")

    cuda_sync(device)
    training_time_sec = time.perf_counter() - fold_start_time
    peak_allocated_mb, peak_reserved_mb = get_peak_memory_mb(device)
    training_steps = max(TOTAL_ITERATIONS - 1, 1)

    final_accuracy = fold_test_accuracies[-1]['test_accuracy'] if fold_test_accuracies else math.nan
    row = {
        'setting_id': setting['setting_id'],
        'setting_label': setting['setting_label'],
        'modalities': setting_modalities_label(setting),
        'num_modalities': len(modality_ids),
        'mfmc_mode': setting['mfmc_mode'],
        'sampled_terms_requested': sampled_terms if sampled_terms is not None else math.nan,
        'mfmc_terms_per_batch': mfmc_terms_per_batch(setting),
        'fold': fold_num,
        'best_test_accuracy': float(best_accuracy),
        'final_test_accuracy': float(final_accuracy) if not math.isnan(final_accuracy) else math.nan,
        'training_time_sec': float(training_time_sec),
        'sec_per_iteration': float(training_time_sec / training_steps),
        'peak_gpu_memory_allocated_mb': peak_allocated_mb,
        'peak_gpu_memory_reserved_mb': peak_reserved_mb,
        'model_parameter_count': model_parameter_count,
        'total_iterations': TOTAL_ITERATIONS,
        'training_steps': training_steps,
        'batch_size': BATCH_SIZE,
        'eval_interval': EVAL_INTERVAL,
        'learning_rate_encoder': LEARNING_RATE_ENCODER,
        'learning_rate_classifier': LEARNING_RATE_CLASSIFIER,
        'cov_beta': COV_BETA,
        'classification_modality': MODALITY_DEFINITIONS[classifier_mid]['label'],
        'device': str(device),
        'timestamp_utc': datetime.now(timezone.utc).isoformat(),
    }

    log_payload = {
        'row': row,
        'mfmc_losses': fold_costs,
        'classifier_losses': fold_classifier_losses,
        'test_accuracies': fold_test_accuracies,
        'selected_term_count_mean': float(np.mean(selected_term_counts)) if selected_term_counts else math.nan,
    }
    log_path = save_fold_log(setting, fold_num, log_payload)
    print(f"Fold log saved to: {log_path}")
    print(f"Fold completed. Best test accuracy: {best_accuracy:.4f}; time: {training_time_sec / 60:.1f} min")

    del encoders, projection_heads, classifier, optimizer_features, optimizer_classifier, trackers
    del train_tensors, test_tensors
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    return row


def get_experiments_to_run():
    if not EXPERIMENT_FILTER:
        return EXPERIMENTS
    requested = {part.strip() for part in EXPERIMENT_FILTER.split(',') if part.strip()}
    return [setting for setting in EXPERIMENTS if setting['setting_id'] in requested]


def run_all_experiments():
    device = select_device()
    existing_df = load_per_fold_results()
    rows = [] if existing_df.empty else existing_df.to_dict('records')

    for setting in get_experiments_to_run():
        for fold_info in fold_splits:
            if completed_fold_exists(pd.DataFrame(rows), setting, fold_info['fold']):
                print(f"Skipping completed {setting['setting_id']} fold {fold_info['fold']}.")
                continue
            row = run_one_fold(setting, fold_info, device)
            rows.append(row)
            per_fold_df = pd.DataFrame(rows)
            per_fold_df.to_csv(PER_FOLD_RESULTS_CSV, index=False)
            aggregate_df = write_aggregate_results(per_fold_df)
            print(f"Updated: {PER_FOLD_RESULTS_CSV}")
            print(f"Updated: {AGGREGATE_RESULTS_CSV}")
            display(aggregate_df)

    per_fold_df = pd.DataFrame(rows)
    if not per_fold_df.empty:
        per_fold_df.to_csv(PER_FOLD_RESULTS_CSV, index=False)
        aggregate_df = write_aggregate_results(per_fold_df)
        return per_fold_df, aggregate_df
    return pd.DataFrame(), pd.DataFrame()


In [ ]:
# Launch cell. Set RUN_EXPERIMENTS=True above, or export MFMC_SCALABILITY_RUN=1,
# when you are ready for the full 4-setting x 5-fold experiment.

if RUN_EXPERIMENTS:
    per_fold_results, aggregate_results = run_all_experiments()
else:
    print('RUN_EXPERIMENTS is False. Training was not launched in this execution.')
    if PER_FOLD_RESULTS_CSV.exists():
        per_fold_results = pd.read_csv(PER_FOLD_RESULTS_CSV)
        aggregate_results = write_aggregate_results(per_fold_results)
        print(f'Loaded existing per-fold results: {PER_FOLD_RESULTS_CSV}')
        display(aggregate_results)
    else:
        per_fold_results = pd.DataFrame()
        aggregate_results = pd.DataFrame()
        print('No existing per-fold result CSV found yet.')


## 7. Results

The two result tables saved by the training cell are:

- `deap_scalability_per_fold_results.csv`
- `deap_scalability_aggregate_results.csv`

The aggregate table reports mean best test accuracy across folds and the standard deviation of the best test accuracies. It also reports average training time, average seconds per iteration, average and maximum peak GPU memory, model parameter count, and the number of MFMC terms evaluated per batch.


In [ ]:
# =============================================================================
# RESULT TABLE DISPLAY
# =============================================================================

if AGGREGATE_RESULTS_CSV.exists():
    aggregate_results = pd.read_csv(AGGREGATE_RESULTS_CSV)
    display_cols = [
        'setting_label',
        'modalities',
        'folds_completed',
        'best_test_accuracy_mean_pm_std',
        'mean_training_time_min',
        'mean_sec_per_iteration',
        'mean_peak_gpu_memory_allocated_mb',
        'model_parameter_count',
        'mfmc_terms_per_batch',
    ]
    display(aggregate_results[display_cols])
else:
    print('Aggregate result table not found yet. Run the training cell first.')


In [ ]:
# =============================================================================
# PUBLICATION-READY FIGURES
# =============================================================================


def ordered_aggregate(df):
    order = {setting['setting_id']: idx for idx, setting in enumerate(EXPERIMENTS)}
    return df.assign(_order=df['setting_id'].map(order)).sort_values('_order').drop(columns=['_order'])


def save_bar_chart(df, y_col, yerr_col, ylabel, title, filename, value_scale=1.0):
    plot_df = ordered_aggregate(df)
    labels = plot_df['setting_label'].tolist()
    values = plot_df[y_col].to_numpy(dtype=float) * value_scale
    errors = None if yerr_col is None else plot_df[yerr_col].to_numpy(dtype=float) * value_scale

    fig, ax = plt.subplots(figsize=(8, 4.5))
    colors = ['#4C78A8', '#72B7B2', '#F58518', '#54A24B'][:len(values)]
    ax.bar(labels, values, yerr=errors, capsize=4, color=colors, edgecolor='black', linewidth=0.6)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis='y', alpha=0.3)
    ax.tick_params(axis='x', rotation=20)
    fig.tight_layout()
    path = FIG_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


def save_combined_summary(df):
    plot_df = ordered_aggregate(df)
    labels = plot_df['setting_label'].tolist()
    x = np.arange(len(labels))
    colors = ['#4C78A8', '#72B7B2', '#F58518', '#54A24B'][:len(labels)]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4.2))

    axes[0].bar(x, plot_df['mean_best_test_accuracy'], yerr=plot_df['std_best_test_accuracy'],
                capsize=4, color=colors, edgecolor='black', linewidth=0.6)
    axes[0].set_ylabel('Best test accuracy')
    axes[0].set_title('Accuracy')
    axes[0].grid(axis='y', alpha=0.3)

    axes[1].bar(x, plot_df['mean_training_time_sec'] / 60.0,
                color=colors, edgecolor='black', linewidth=0.6)
    axes[1].set_ylabel('Training time per fold (min)')
    axes[1].set_title('Wall-clock time')
    axes[1].grid(axis='y', alpha=0.3)

    axes[2].bar(x, plot_df['mean_peak_gpu_memory_allocated_mb'],
                color=colors, edgecolor='black', linewidth=0.6)
    axes[2].set_ylabel('Peak GPU memory (MB)')
    axes[2].set_title('GPU memory')
    axes[2].grid(axis='y', alpha=0.3)

    for ax in axes:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=20, ha='right')

    fig.tight_layout()
    path = FIG_DIR / 'deap_scalability_combined_summary.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


def save_terms_vs_compute(df):
    plot_df = ordered_aggregate(df)
    labels = plot_df['setting_label'].tolist()

    fig, ax1 = plt.subplots(figsize=(8, 4.5))
    ax1.plot(labels, plot_df['mean_sec_per_iteration'], marker='o', color='#4C78A8', label='sec/iteration')
    ax1.set_ylabel('Seconds per iteration')
    ax1.set_title('MFMC terms and compute cost')
    ax1.grid(axis='y', alpha=0.3)
    ax1.tick_params(axis='x', rotation=20)

    ax2 = ax1.twinx()
    ax2.plot(labels, plot_df['mfmc_terms_per_batch'], marker='s', color='#F58518', label='terms/batch')
    ax2.set_ylabel('MFMC terms per batch')

    lines, line_labels = ax1.get_legend_handles_labels()
    lines2, line_labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, line_labels + line_labels2, loc='upper left')

    fig.tight_layout()
    path = FIG_DIR / 'deap_scalability_terms_vs_compute.png'
    fig.savefig(path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    print(f'Saved {path}')


def generate_result_figures():
    if not AGGREGATE_RESULTS_CSV.exists():
        print('No aggregate results found. Run the training cell before generating result figures.')
        return
    df = pd.read_csv(AGGREGATE_RESULTS_CSV)
    save_bar_chart(
        df,
        y_col='mean_best_test_accuracy',
        yerr_col='std_best_test_accuracy',
        ylabel='Best test accuracy',
        title='DEAP MFMC scalability: accuracy',
        filename='deap_scalability_accuracy_bar.png',
    )
    save_bar_chart(
        df,
        y_col='mean_training_time_sec',
        yerr_col='std_training_time_sec',
        ylabel='Training time per fold (min)',
        title='DEAP MFMC scalability: training time',
        filename='deap_scalability_training_time_bar.png',
        value_scale=1.0 / 60.0,
    )
    save_bar_chart(
        df,
        y_col='mean_peak_gpu_memory_allocated_mb',
        yerr_col=None,
        ylabel='Peak GPU memory (MB)',
        title='DEAP MFMC scalability: peak GPU memory',
        filename='deap_scalability_peak_gpu_memory_bar.png',
    )
    save_combined_summary(df)
    save_terms_vs_compute(df)


generate_result_figures()


## 8. Complexity and Memory Analysis

Let `M` be the number of modalities, `K` the embedding dimension, `B` the batch size, `H` the projection hidden width, and `T` the number of evaluated leave-one-out terms per batch. In this implementation, `K = 128`, `H = 512`, and `T = M` for full-sum MFMC. For sampled MFMC, `T = s`, where `s` is the sampled term count.

Practical scaling summary:

- Encoder compute grows roughly linearly with the number of modalities, because all `M` encoders are evaluated each batch.
- Projection and MFMC objective compute grow with the number of evaluated leave-one-out terms. Each term uses a projection from `(M - 1)K` to `K`, covariance estimates with cost around `O(B K^2)`, and matrix inverses/products around `O(K^3)`.
- Full-sum objective cost therefore trends like `O(M * (B (M - 1) K H + B K^2 + K^3))` plus encoder cost.
- Sampled objective cost trends like `O(s * (B (M - 1) K H + B K^2 + K^3))` plus the same `M` encoder cost.
- Tracker memory is small and scales as `O(3 M K^2)` float elements for Rx, Ry, and Pxy per leave-one-out target. Activation memory usually dominates and grows with the number of encoders plus the number of projection terms evaluated in the current batch.
- Projection parameters grow faster than linearly because there are `M` heads and each head sees `(M - 1)K` input features, so the first projection layer contributes approximately `O(M (M - 1) K H)` parameters.


In [ ]:
# =============================================================================
# COMPLEXITY AND MODEL SIZE SUMMARIES
# =============================================================================


def build_complexity_summary():
    rows = []
    for setting in EXPERIMENTS:
        num_modalities = len(setting['modalities'])
        terms = mfmc_terms_per_batch(setting)
        projection_input_dim = (num_modalities - 1) * FEATURE_DIM
        rows.append({
            'setting_id': setting['setting_id'],
            'setting_label': setting['setting_label'],
            'modalities': setting_modalities_label(setting),
            'num_modalities_M': num_modalities,
            'embedding_dim_K': FEATURE_DIM,
            'batch_size_B': BATCH_SIZE,
            'projection_hidden_dim_H': PROJECTION_HIDDEN_DIM,
            'mfmc_terms_evaluated_T': terms,
            'full_leave_one_out_terms': num_modalities,
            'projection_input_dim': projection_input_dim,
            'covariance_tracker_float_elements': 3 * num_modalities * FEATURE_DIM * FEATURE_DIM,
            'relative_projection_term_index': terms * (num_modalities - 1),
            'relative_mfmc_matrix_term_index': terms,
            'objective_type': setting['mfmc_mode'],
        })
    df = pd.DataFrame(rows)
    df.to_csv(COMPLEXITY_SUMMARY_CSV, index=False)
    return df


def build_model_size_summary():
    rows = []
    device = torch.device('cpu')
    for setting in EXPERIMENTS:
        encoders, projection_heads, classifier = create_models_for_setting(setting, device)
        encoder_params = count_parameters(encoders)
        projection_params = count_parameters(projection_heads)
        classifier_params = count_parameters(classifier)
        rows.append({
            'setting_id': setting['setting_id'],
            'setting_label': setting['setting_label'],
            'modalities': setting_modalities_label(setting),
            'num_modalities': len(setting['modalities']),
            'mfmc_terms_per_batch': mfmc_terms_per_batch(setting),
            'encoder_parameter_count': encoder_params,
            'projection_parameter_count': projection_params,
            'classifier_parameter_count': classifier_params,
            'total_model_parameter_count': encoder_params + projection_params + classifier_params,
        })
        del encoders, projection_heads, classifier
        gc.collect()
    df = pd.DataFrame(rows)
    df.to_csv(MODEL_SIZE_SUMMARY_CSV, index=False)
    return df


complexity_df = build_complexity_summary()
model_size_df = build_model_size_summary()
print(f'Saved {COMPLEXITY_SUMMARY_CSV}')
print(f'Saved {MODEL_SIZE_SUMMARY_CSV}')
display(complexity_df)
display(model_size_df)


In [ ]:
# Optional theoretical plot that does not require completed training results.
fig, ax1 = plt.subplots(figsize=(8, 4.5))
plot_df = complexity_df.copy()
labels = plot_df['setting_label'].tolist()
ax1.plot(labels, plot_df['mfmc_terms_evaluated_T'], marker='o', color='#4C78A8', label='MFMC terms per batch')
ax1.set_ylabel('MFMC terms per batch')
ax1.set_title('Theoretical leave-one-out term count')
ax1.grid(axis='y', alpha=0.3)
ax1.tick_params(axis='x', rotation=20)

ax2 = ax1.twinx()
ax2.plot(labels, plot_df['relative_projection_term_index'], marker='s', color='#F58518', label='relative projection index')
ax2.set_ylabel('Terms x (M - 1)')

lines, line_labels = ax1.get_legend_handles_labels()
lines2, line_labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines + lines2, line_labels + line_labels2, loc='upper left')
fig.tight_layout()
path = FIG_DIR / 'deap_scalability_theoretical_terms.png'
fig.savefig(path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f'Saved {path}')


## 9. Key Conclusions

Run the training cell to populate the final numerical conclusions. The cell below turns the aggregate result table into a short appendix-ready summary that focuses on the requested questions: accuracy stability, runtime growth, peak memory growth, and the 5-modal sampled tradeoff relative to 5-modal full-sum.


In [ ]:
# =============================================================================
# FINAL SUMMARY TEXT
# =============================================================================


def format_pct(value):
    return f'{100.0 * value:.2f}%'


def build_final_summary():
    if not AGGREGATE_RESULTS_CSV.exists():
        return Markdown(
            'Final numerical summary is pending. Run the training cell to create '
            '`deap_scalability_aggregate_results.csv`, then rerun this cell.'
        )

    df = ordered_aggregate(pd.read_csv(AGGREGATE_RESULTS_CSV))
    lines = ['### Final summary']
    for _, row in df.iterrows():
        lines.append(
            f"- {row['setting_label']}: best test accuracy "
            f"{format_pct(row['mean_best_test_accuracy'])} +/- "
            f"{format_pct(row['std_best_test_accuracy'])}, "
            f"training time {row['mean_training_time_sec'] / 60.0:.1f} min/fold, "
            f"peak GPU memory {row['mean_peak_gpu_memory_allocated_mb']:.0f} MB, "
            f"{int(row['mfmc_terms_per_batch'])} MFMC terms/batch."
        )

    by_id = {row['setting_id']: row for _, row in df.iterrows()}
    if '3mod_full' in by_id and '5mod_full' in by_id:
        acc_delta = by_id['5mod_full']['mean_best_test_accuracy'] - by_id['3mod_full']['mean_best_test_accuracy']
        time_ratio = by_id['5mod_full']['mean_training_time_sec'] / by_id['3mod_full']['mean_training_time_sec']
        mem_ratio = by_id['5mod_full']['mean_peak_gpu_memory_allocated_mb'] / by_id['3mod_full']['mean_peak_gpu_memory_allocated_mb']
        lines.append(
            f"- From 3-modal full-sum to 5-modal full-sum, mean best accuracy changes by "
            f"{100.0 * acc_delta:+.2f} percentage points, runtime changes by {time_ratio:.2f}x, "
            f"and peak allocated GPU memory changes by {mem_ratio:.2f}x."
        )

    if '5mod_full' in by_id and '5mod_sampled' in by_id:
        sampled = by_id['5mod_sampled']
        full = by_id['5mod_full']
        acc_delta = sampled['mean_best_test_accuracy'] - full['mean_best_test_accuracy']
        time_ratio = sampled['mean_training_time_sec'] / full['mean_training_time_sec']
        mem_ratio = sampled['mean_peak_gpu_memory_allocated_mb'] / full['mean_peak_gpu_memory_allocated_mb']
        lines.append(
            f"- Relative to 5-modal full-sum, 5-modal sampled changes mean best accuracy by "
            f"{100.0 * acc_delta:+.2f} percentage points, uses {time_ratio:.2f}x runtime, "
            f"and uses {mem_ratio:.2f}x peak allocated GPU memory."
        )

    return Markdown('\n'.join(lines))


display(build_final_summary())
